# 평가 Gold Set 생성 개요

- **목적**

2만 개 stratified sample 도서 DB를 활용하여 retrieval/rerank 모델 실험용 평가셋을 생성한다.

현재 단계에서는 전체 사람 검수가 어렵기 때문에, LLM judge를 활용해 후보 도서에 대한 relevance pseudo-label을 생성하고, 이를 기반으로 1차 gold candidate set을 구성한다.

---

- **입력 데이터**

 1. `scenario_data.json`

- 경로: `evaluation/dataset/scenario_data.json`
- 구성:
  - `onboarding_data`: 사용자 장기 선호 정보
  - `session_data`: 현재 대화 세션에서 추출된 사용자 요청 정보

 2. 도서 DB

- 약 2만 개 stratified sample 도서 데이터
- BM25 / Dense / Hybrid 검색 대상
- ISBN 기준 도서 메타데이터 조회 대상

---

- **생성 과정**

1. `session_data`와 `onboarding_data`를 기반으로 검색 질의를 구성한다.

2. 도서 DB를 대상으로 BM25, Dense, Hybrid retriever를 각각 실행하여 Top-40 후보를 추출한다.

3. 세 retriever 결과를 merge하고, ISBN 기준으로 중복 제거한다.

4. 중복 제거된 ISBN을 기준으로 DB에서 도서 메타데이터를 조회한다.

5. LLM judge가 각 후보 도서의 relevance를 `0~3 grade`로 판단한다.

   - `session_data`를 우선 기준으로 사용한다.
   - `use_onboarding=true`인 경우 `onboarding_data`를 보조 기준으로 반영한다.
   - `session_data`와 `onboarding_data`가 충돌하면 `session_data`를 우선한다.
   - retrieval rank, score, source는 LLM judge 입력에서 제외한다.
   - 단, `retrieval_info`는 분석 및 hard negative 추출을 위해 출력 데이터에는 저장한다.

6. `relevance_grade`를 기반으로 label을 생성한다.

```python
binary_label = 1 if relevance_grade >= 2 else 0
```

출력 항목 예시:

- `relevance_grade`: 0, 1, 2, 3
- `binary_label`: 0 또는 1
- `retrieval_info`: source/rank/score 정보


In [12]:
import os
os.chdir("/home/ubuntu/work/somin/backend")
os.getcwd()


'/home/ubuntu/work/somin/backend'

In [13]:
from app.modules.RAG.retriever import (
                                    full_bm25_with_onboarding,
                                    full_dense_with_onboarding,
                                    full_hybrid_with_onboarding,
                                    make_onboarding_result
                                    )
import json
import uuid
import time
import random
import requests
from pathlib import Path
from tqdm import tqdm
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

# Retriever 후보 Pool

In [14]:
SCENARIO_PATH = Path("/home/ubuntu/work/somin/evaluation/dataset/scenario_data.json")

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

print(f"시나리오 수: {len(scenarios)}")


def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query


In [ ]:
def deduplicate_results(results_list, key="isbn"):
    merged = []
    seen = set()

    for results in results_list:
        for item in results:

            item_key = item.get(key)

            if item_key in seen:
                continue

            seen.add(item_key)
            merged.append(item)

    return merged


# =========================
# full 결과 통합
# =========================

full_merged = deduplicate_results([
    re1_bm25,
    re1_dense,
    re1_hybrid
])

full_result_json = {
    "query": sample_result,
    "results": full_merged
}

#with open("full_results.json", "w", encoding="utf-8") as f:
#    json.dump(full_result_json, f, ensure_ascii=False, indent=2)

def simplify_item(item, source_name):
    return {
        "isbn": item.get("isbn"),
        "title": item.get("title"),
        "author": item.get("author"),
        "publisher": item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page": item.get("page"),

        "large_cate": item.get("large_cate"),
        "mid_cate": item.get("mid_cate"),
        "small_cate": item.get("small_cate"),

        "book_intro": item.get("book_intro"),
        "book_index": item.get("book_index"),

        "review_count": item.get("review_count"),
        "review_text": item.get("review_text"),

        # "retrieval_source": source_name,
        # "retrieval_rank": item.get("rank"),
        # "retrieval_score": item.get("score")
    }


def merge_with_sources(result_groups, key="isbn"):
    merged = {}

    for source_name, results in result_groups.items():
        for item in results:
            item_key = item.get(key)

            if item_key is None:
                continue

            simplified = simplify_item(item, source_name)

            if item_key not in merged:
                base = {
                    k: v for k, v in simplified.items()
                    if not k.startswith("retrieval_")
                }

                base["retrieval_info"] = []
                merged[item_key] = base

            merged[item_key]["retrieval_info"].append({
                "source": source_name,
                "rank": item.get("rank"),
                "score": item.get("score")
            })

    return list(merged.values())

full_books = merge_with_sources({
    "bm25": re1_bm25,
    "dense": re1_dense,
    "hybrid": re1_hybrid
})

#with open("full_books_for_labeling.json", "w", encoding="utf-8") as f:
#   json.dump(full_books, f, ensure_ascii=False, indent=2)


# LLM-Judge labeling

In [ ]:
URL = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
CLOVA_API_KEY = settings.CLOVA_API_KEY

SYSTEM_PROMPT = """
당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

입력으로는 다음 두 정보가 주어집니다.

1. request: 사용자의 검색 요청 전체
- keyword_query
- semantic_query
- filters
- constraints
- score_boost
- session_signals
- onboarding_signals

2. book: 추천 후보 도서 정보

[판정 기준]
- 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
- session_signals를 최우선 기준으로 판단하세요.
- onboarding_signals는 session_signals와 충돌하지 않는 경우에만 보조적으로 반영하세요.
- session_signals와 onboarding_signals가 충돌하면 session_signals를 우선하세요.
- 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
- 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
- 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 높은 점수를 부여하세요.
- 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 낮은 점수를 부여하세요.
- filters, constraints는 명시 조건으로 보고 우선 고려하세요.
- soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 높은 점수를 부여할 수 있습니다.
- 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 낮은 점수를 부여하세요.
- 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 낮은 점수를 부여하세요.
- retrieval rank, score, source 정보는 판단에 사용하지 마세요.
- 반드시 JSON만 출력하세요. 설명 문장, markdown, 코드블록은 출력하지 마세요.

[Grade 기준]
3 = 매우 적합. 현재 요청의 핵심 의도와 조건에 잘 부합함.
2 = 적합. 핵심 의도는 맞지만 일부 조건이 약함.
1 = 부분 관련. 관련성은 있으나 정답 도서로 보기에는 약함.
0 = 부적합. 현재 요청의 핵심 의도와 맞지 않음.

출력 형식:
{
  "relevance_grade": 0 또는 1 또는 2 또는 3
  "reason": "판정 근거를 간략히 설명하는 텍스트"
}
"""

def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream"
    }


def make_book_for_judge(book):
    """
    LLM judge 입력에서 retrieval 관련 정보를 제거한다.
    원본 book의 retrieval_info는 최종 저장 데이터에는 유지한다.
    """
    exclude_keys = {
        "retrieval_info",
        "rank",
        "score",
        "source",
        "retrieval_source",
        "retrieval_rank",
        "retrieval_score",
        "bm25_score",
        "dense_score",
        "hybrid_score",
        "main_score",
        "onboarding_score",
        "disliked_penalty",
        "disliked_similarity"
    }

    return {
        k: v for k, v in book.items()
        if k not in exclude_keys and not str(k).startswith("retrieval_")
    }


def parse_hcx_response(text):
    """
    HCX 응답이 SSE 형식 또는 일반 JSON 형식으로 올 수 있어 둘 다 처리한다.
    """
    # 1. SSE 형식 처리
    if "event:" in text:
        last_data = None

        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = None
            data_text = None

            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()

            if event_type == "result" and data_text:
                last_data = data_text

        if last_data is not None:
            data = json.loads(last_data)
            return data["message"]["content"]

    # 2. 일반 JSON 응답 처리
    try:
        data = json.loads(text)

        if "result" in data:
            return data["result"]["message"]["content"]

        if "message" in data:
            return data["message"]["content"]

    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_grade(llm_text):
    """
    파싱 실패를 grade=0으로 처리하지 않는다.
    실패 시 relevance_grade=None, label_status='parse_failed'로 반환한다.
    """
    try:
        obj = json.loads(llm_text)
        grade = obj.get("relevance_grade", obj.get("grade", obj.get("label")))

        if grade is None:
            return None, "parse_failed", "missing relevance_grade"

        grade = int(grade)

        if grade not in [0, 1, 2, 3]:
            return None, "parse_failed", f"invalid relevance_grade: {grade}"

        return grade, "success", None

    except Exception as e:
        return None, "parse_failed", str(e)


def grade_to_binary(grade):
    if grade is None:
        return None
    return 1 if grade >= 2 else 0


def make_user_prompt(request_data, book):
    payload = {
        "request": request_data,
        "book": make_book_for_judge(book)
    }

    return json.dumps(payload, ensure_ascii=False, indent=2)


def label_one_book(request_data, book, max_retries=5):
    query_id = request_data.get("query_id", "sample_result")
    user_prompt = make_user_prompt(request_data, book)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "topP": 0.1,
        "topK": 0,
        "max_tokens": 100,
        "temperature": 0.0,
        "repetitionPenalty": 1.0,
        "includeAiFilters": False,
        "seed": 42
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            res = requests.post(
                URL,
                headers=make_headers(),
                json=body,
                timeout=60
            )

            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                grade, label_status, parse_error = extract_grade(llm_text)
                binary_label = grade_to_binary(grade)

                return {
                    **book,
                    "query_id": query_id,
                    "relevance_grade": grade,
                    "binary_label": binary_label,
                    "label_status": label_status,
                    "llm_raw_output": llm_text,
                    "llm_error": parse_error
                }

            last_error = f"{res.status_code} / {res.text[:500]}"

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)

            # 400/401 등은 재시도해도 해결되지 않을 가능성이 높으므로 즉시 실패 처리
            return {
                **book,
                "query_id": query_id,
                "relevance_grade": None,
                "binary_label": None,
                "label_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": last_error
            }

        except Exception as e:
            last_error = str(e)

            if attempt == max_retries - 1:
                return {
                    **book,
                    "query_id": query_id,
                    "relevance_grade": None,
                    "binary_label": None,
                    "label_status": "api_failed",
                    "llm_raw_output": None,
                    "llm_error": last_error
                }

            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            print(f"[재시도 {attempt + 1}/{max_retries}] {wait:.1f}초 대기 → {last_error}")
            time.sleep(wait)


In [6]:
labeled_full_books = []

jsonl_path      = Path("/home/ubuntu/work/somin/evaluation/dataset/goldset_all_scenarios.jsonl")
final_json_path = Path("/home/ubuntu/work/somin/evaluation/dataset/goldset_all_scenarios.json")
jsonl_path.parent.mkdir(parents=True, exist_ok=True)

# 이미 처리된 (query_id, isbn)은 재실행 시 skip한다.
# 처음부터 다시 만들려면 기존 jsonl/json 파일을 삭제한 뒤 실행한다.
processed_keys: set = set()

if jsonl_path.exists():
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                isbn = item.get("isbn")
                if isbn:
                    processed_keys.add((item.get("query_id"), isbn))
                    labeled_full_books.append(item)
            except json.JSONDecodeError:
                continue

print(f"이미 처리된 도서 수: {len(processed_keys)}")

with jsonl_path.open("a", encoding="utf-8") as jsonl_file:
    for scenario in tqdm(scenarios, desc="시나리오"):
        sample_result = extract_rag_query(scenario)
        query_id      = sample_result["query_id"]

        # 검색
        re_bm25   = full_bm25_with_onboarding(sample_result, size=40)
        re_dense  = full_dense_with_onboarding(sample_result, size=40)
        re_hybrid = full_hybrid_with_onboarding(sample_result, size=40)

        # 머지 & 중복 제거
        full_books = merge_with_sources({
            "bm25":   re_bm25,
            "dense":  re_dense,
            "hybrid": re_hybrid,
        })

        # 라벨링
        for book in tqdm(full_books, desc=f"  {query_id}", leave=False):
            isbn = book.get("isbn")
            key  = (query_id, isbn)
            if key in processed_keys:
                continue

            labeled = label_one_book(request_data=sample_result, book=book)
            labeled_full_books.append(labeled)
            processed_keys.add(key)

            jsonl_file.write(json.dumps(labeled, ensure_ascii=False) + "\n")
            jsonl_file.flush()

# 최종 JSON 저장
with final_json_path.open("w", encoding="utf-8") as f:
    json.dump(labeled_full_books, f, ensure_ascii=False, indent=2)

print(f"\nJSONL 저장 완료: {jsonl_path}")
print(f"JSON  저장 완료: {final_json_path}")

# 통계
from collections import defaultdict
stats = defaultdict(lambda: defaultdict(int))
for item in labeled_full_books:
    stats[item.get("query_id", "unknown")][item.get("relevance_grade")] += 1

print("\n[query_id별 relevance_grade 분포]")
print(f"{'query_id':>10}  {'grade 0':>7}  {'grade 1':>7}  {'grade 2':>7}  {'grade 3':>7}  {'total':>7}")
print("-" * 55)
for qid in sorted(stats):
    g = stats[qid]
    total = sum(g.values())
    print(f"{qid:>10}  {g[0]:>7}  {g[1]:>7}  {g[2]:>7}  {g[3]:>7}  {total:>7}")


# 결과 확인

In [ ]:
# 저장 경로
OUTPUT_JSONL_PATH = "/home/ubuntu/work/somin/evaluation/dataset/full_books_labeled.jsonl"

# JSONL 로드
rows = []

with open(OUTPUT_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print(f"전체 라벨링 결과 수: {len(df):,}")
df.head()

전체 라벨링 결과 수: 86


,isbn,title,author,publisher,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,review_count,review_text,retrieval_info,query_id,relevance_grade,binary_label,label_status,llm_raw_output,llm_error
0,9788973379552,화요일의 동물원 - 꿈을 찾는 이들에게 보내는 희망과 위안의 메시지,"박민정 저, 사진",해냄,2008-07-07,231,[시/에세이],[나라별 에세이],[한국에세이],"4년 동안 160번의 화요일을 동물원에서 보낸 후 사랑과 행복, 그리고 영혼의 깨달...",,0,,"[{'source': 'bm25', 'rank': 1, 'score': 0.7870...",sample_result,2,1,success,"{\n ""relevance_grade"": 2\n}",None
1,9788997585458,사랑하라 다시 한번 더,김지연 저,마음세상,2012-08-20,306,"[시/에세이, 자기계발]","[나라별 에세이, 테마에세이]","[사랑/연애에세이, 한국에세이]",마음이 따뜻해지는 힐링 메시지\n누구나 설레는 마음을 안고 인생을 시작한다. 하지만...,,0,,"[{'source': 'bm25', 'rank': 2, 'score': 0.7}, ...",sample_result,1,0,success,"{\n ""relevance_grade"": 1\n}",None
2,9788926728079,대영반 5,위상 저,파피루스(디앤씨미디어),2012-12-26,309,[소설],"[장르소설, 한국소설]",[무협소설],최대 연재 사이트에서 6편 연재만에 골든 베스트 입성!\n골든 베스트 1위! 선호작...,,0,,"[{'source': 'bm25', 'rank': 3, 'score': 0.6346...",sample_result,1,0,success,"{\n ""relevance_grade"": 1\n}",None
3,9788996107095,작은꿈이 꽃필때 - 마음이 편안해지고 싶을 때 읽는 책,장성철,생각의정원,2015-06-01,266,[시/에세이],[],[],『작은 꿈이 꽃필때』는 아름다운 마우이 섬에서 자연을 벗 삼아 가야할 길과 보아야 ...,,0,,"[{'source': 'bm25', 'rank': 4, 'score': 0.5615...",sample_result,1,0,success,"{\n ""relevance_grade"": 1\n}",None
4,9788971070499,김소월 - 비극적 삶과 문학적 형상화,김영철 저,건국대학교출판부,1994-04-01,116,"[소설, 대학교재/전문서적]",[한국문학론],[한국작가론],민중 속에서 민중의 사랑을 받으며 애송되어 온 시가 소월의 시다. 자연의 숨결과 민...,,0,,"[{'source': 'bm25', 'rank': 5, 'score': 0.5181...",sample_result,1,0,success,"{\n ""relevance_grade"": 1\n}",None


In [8]:
status_dist = df["label_status"].value_counts(dropna=False).reset_index()
status_dist.columns = ["label_status", "count"]
status_dist["ratio"] = status_dist["count"] / len(df)

status_dist

,label_status,count,ratio
0,success,86,1.0


In [9]:
success_df = df[df["label_status"] == "success"].copy()

print(f"성공 라벨 수: {len(success_df):,}")
print(f"실패/제외 수: {len(df) - len(success_df):,}")

성공 라벨 수: 86
실패/제외 수: 0


In [10]:
grade_dist = (
    success_df["relevance_grade"]
    .value_counts()
    .sort_index()
    .reset_index()
)

grade_dist.columns = ["relevance_grade", "count"]
grade_dist["ratio"] = grade_dist["count"] / len(success_df)

grade_dist

,relevance_grade,count,ratio
0,0,7,0.081395
1,1,61,0.709302
2,2,15,0.174419
3,3,3,0.034884
